# In this notebook we aim to answer the following questions:

### ✔ 1. Is there a country-level BPM preference?
### ✔ 2. Do faster songs have higher popularity?
### ✔ 3. Do louder songs have higher popularity?

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('darkgrid')

# Load the dataset with audio features and normalized popularity created in eu_normalized_popularity_top200.ipynb
df_charts= pd.read_csv("../data/eu_top200_with_custom_popularity.csv")
df_charts.info()

## 🔎  Univariate Analysis: data distribution of key audio features


#### We'll create a dataset by removing duplicate tracks from the top200 charts dataset loaded in the previous step, since the audio features are the same for track_id:

In [ ]:
df_charts.groupby('track_id').value_counts()

In [ ]:
columns_names = ['track_id','normalized_popularity','tempo','danceability','energy','loudness']
df_tracks = df_charts[columns_names].drop_duplicates(subset=['track_id'])
df_tracks.value_counts()

In [ ]:
# Analysis of the distribution of key audio features
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

df_tracks['tempo'].hist(bins=50, ax=axes[0, 0])
axes[0, 0].set_title('Tempo Distribution')

df_tracks['loudness'].hist(bins=50, ax=axes[0, 1])
axes[0, 1].set_title('Loudness Distribution')

df_tracks['danceability'].hist(bins=50, ax=axes[1, 0])
axes[1, 0].set_title('Danceability Distribution')

df_tracks['energy'].hist(bins=50, ax=axes[1, 1])
axes[1, 1].set_title('Energy Distribution')

plt.tight_layout()
plt.show()

## Summary

##### 🥁 Tempo
Clear multimodal distribution — peaks at ~95, ~120–130, ~150 BPM.
Most songs fall between ~90–150 BPM;    few very slow or very fast tracks.
We can see at least 3–4 distinct peaks:

| Peak | BPM | Likely genre |
|---|---|---|
| Peak 1 | ~80 BPM | Hip-hop, R&B |
| Peak 2 | ~100 BPM | Reggaeton, trap |
| Peak 3 | ~120–130 BPM | Pop, dance, EDM (dominant) |
| Peak 4 | ~150–160 BPM | Drum & bass, fast EDM, or half-time of ~75-80 BPM |

The **120–130 BPM peak is clearly dominant**, but the other modes are visible and meaningful.

##### 🥁 Loudness
Strong left skew; values clustered near the loud end (~-8 to -4 dB), peak ~-5 dB.
Few very quiet tracks; charts dominated by loud, highly compressed productions [loudness‑war effect](https://en.wikipedia.org/wiki/Loudness_war).
##### 🥁 Danceability
Approximately bell‑shaped, slightly left‑skewed; peak around 0.65–0.8.
Most chart tracks are fairly danceable; low danceability is rare.
##### 🥁 Energy
Left‑skewed with a broad plateau; peak around 0.6–0.8.
High‑energy tracks are common but distribution is wider than danceability (more variety).


## 🇪🇺 1. Country-level Tempo/BPM preference analysis in EU

####  Not-weighted distribution, analyze the tempo distribution based only on the presence of a track in the charts

In [ ]:
# Load the European regions mapping from CSV
eu_regions = pd.read_csv('../data/european_countries.csv')
region_map = dict(zip(eu_regions['country'], eu_regions['region']))

# Map each country (df_charts['country']) to its European macro-region
df_charts['eu_region'] = df_charts['country'].map(region_map)

palette = {
    'Northern Europe': '#2ecc71',
    'Western Europe': '#3498db',
    'Southern Europe': '#e74c3c',
    'Eastern Europe': '#f39c12',
}

plt.figure(figsize=(14, 6))
sns.kdeplot(data=df_charts, x='tempo', hue='eu_region',
            fill=True, alpha=0.15, palette=palette, linewidth=2, common_norm=False)
plt.xlabel('Tempo (BPM)')
plt.title('Tempo Distribution by European Region')
sns.move_legend(plt.gca(), loc='upper left', bbox_to_anchor=(1, 1), title='European Region')
plt.tight_layout()
plt.show()

## Summary

**Overall pattern**
- All regions share two main tempo peaks:
  - **~95–105 BPM** (mid-tempo, groove-oriented)
  - **~120–130 BPM** (dance / club standard)
- This indicates a strong, shared European tempo backbone.

**Regional characteristics**
- **Southern Europe**
  - Fastest overall tempo profile
  - Strong high-BPM tail (≈160–190+ BPM)
  - Emphasis on energetic, rhythm-forward tracks

- **Western Europe**
  - Most pronounced peak at **~125 BPM**
  - Highly standardized, club-oriented tempos
  - Moderate presence of very fast tracks

- **Northern Europe**
  - Most balanced distribution
  - Broad spread between **100–140 BPM**
  - Fewer extreme fast tempos

- **Eastern Europe**
  - Slightly faster than West/North on average
  - Strong presence around **130–150 BPM**
  - Less extreme high-BPM tail than Southern Europe


#### Streams-weighted tempo distribution

Weigh tempo by `streams` so each track contributes proportionally to how many times it was streamed.

In [ ]:

plt.figure(figsize=(14, 6))
for region in df_charts['eu_region'].unique():
    regional_df = df_charts.loc[df_charts['eu_region'] == region]
    # KDE weighted by streams for a cleaner per-region density
    sns.kdeplot(data=regional_df, x='tempo', weights=regional_df['streams'].values, bw_adjust=1.0,
                fill=False, alpha=0.9, linewidth=2, common_norm=False, label=region)

# global (all-region) streams-weighted KDE
sns.kdeplot(data=df_charts, x='tempo', weights=df_charts['streams'].values, bw_adjust=1.0,
            color='k', linestyle='--', fill=False, common_norm=False, label='All regions (streams-weighted)')

plt.xlabel('Tempo (BPM)')
plt.ylabel('Streams-weighted density')
plt.title('Streams-weighted Tempo Distribution by European Region (KDE)')
plt.legend(bbox_to_anchor=(1, 1), title='Region')
plt.tight_layout()
plt.show()

## Summary
### Streams-weighted vs Unweighted Tempo Distributions (General)

- **Unweighted distributions** reflect the diversity of produced music, showing a wide range of tempos.
- **Streams-weighted distributions** reflect listening behavior, where a smaller set of tempos dominates.
- Weighting by streams compresses tempo diversity, amplifying a few highly popular tempo ranges.
- **Extreme tempos** (very slow or very fast) remain present but account for relatively little overall listening.


## 🔗‍️ 2. Analyze correlation matrix of key audio features (identify relationship between pairs of variables)

In [ ]:
# Correlation matrix for audio features vs normalized_popularity
features = ['tempo', 'loudness', 'energy', 'danceability', 'normalized_popularity']
corr = df_tracks[features].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm', center=0)
plt.title('Correlation: Audio Features vs Normalized Popularity')
plt.tight_layout()
plt.show()
#todo: divide into genres and compare correlations

## Summary
### Relationship Between Audio Features and Popularity

Pearson correlation analysis indicates **no meaningful linear relationship between tempo and normalized popularity**(r ≈ −0.02), suggesting that faster songs are not associated with higher popularity. 

Loudness shows a weak positive correlation with popularity (r ≈ 0.08); however, the magnitude of this association is small and unlikely to be practically significant. 

Overall, these results suggest that basic audio features such as tempo and loudness alone do not strongly explain variation in song popularity.
